# D48 Async

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.10
제작 : 박광석 (모두의연구소)

# 비동기 처리와 에러 핸들링

```
[동기 API 서버 : 모델 추론에 3초 소요]

요청 A (0초) → 추론 → 응답 (3초)
요청 B (0초) → 대기 → 추론 → 응답 (6초)
요청 C (0초) → 대기 → 대기 → 추론 → 응답 (9초)
```

I/O 바운드 vs CPU 바운드:
- I/O 바운드 (DB, 외부 API) → 비동기로 효율 극대화 가능
- **CPU 바운드 (모델 추론)** → 단순 async/await만으로는 해결 안 됨 → `run_in_executor` 필요

`async def` 안에서 동기 작업을 수행하면 이벤트 루프가 멈춰 다른 요청을 처리할 수 없습니다.

## 실행 환경

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 내용 |
|---|---|
| 기준 환경 | 내 PC의 주피터 또는 코랩. GPU 없이 CPU로 돕니다 |
| 준비물 | `app` 폴더, 6장의 `models/mnist_state_dict.pth` (없으면 6.0절 셀이 만듭니다) |
| 커널 재시작 | 3.1절, 4.2절, 6.2절 서버 셀 앞 |
| 포트 | 1\~6장 서버 8000, 7장 서버는 8048부터 빈 포트 |
| 오래 걸리는 셀 | 3.2절과 4.2절의 blocking 동시 요청 각 9초 안팎, 7장 전체 약 30초 |

</div>

🖥 **코랩에서 할 때**  
코랩 런타임이 새로 뜨면 만든 파일이 사라집니다. REST API 노트에서 구글 드라이브에 만든 `model-serving-course` 폴더로 들어가서 시작합니다. 내 PC에서는 이 셀을 건너뛰고 `model-serving-course` 폴더에서 노트북을 엽니다.

세션을 다시 시작하면 작업 폴더가 `/content`로 돌아갑니다. 그때마다 이 셀부터 다시 실행합니다.

In [ ]:
# 코랩에서만 실행합니다. 내 PC에서는 건너뜁니다
import os
from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/model-serving-course"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print("작업 폴더 :", os.getcwd())

REST API 노트와 FastAPI 기초 노트를 드라이브의 `model-serving-course` 폴더에서 했다면 `app/`, `models/` 폴더와 모델 파일이 그대로 있습니다. `app` 폴더가 없으면 3.1절의 `%%writefile`이 `FileNotFoundError`로 멈춥니다. 그때는 3.1절 전에 `os.makedirs("app", exist_ok=True)`를 한 번 실행하십시오.

커널 재시작은 코랩에서 [런타임] - [세션 다시 시작]입니다.

## 1. 동기 vs 비동기: 왜 API 서버에서 중요한가?

> **학습 목표**
> - 동기(Synchronous)와 비동기(Asynchronous)의 차이를 명확히 구분할 수 있습니다.
> - 모델 추론 API에서 동기 처리가 왜 심각한 병목이 되는지 설명할 수 있습니다.
> - FastAPI가 내부적으로 요청을 어떻게 처리하는지 흐름을 이해합니다.

### 1.1 지금까지의 흐름 복습

FastAPI 기초 노트에서 완성한 API는 정상적으로 동작합니다.
한 명이 요청을 보내면, 정확한 결과가 돌아옵니다.

그런데 **동시에 여러 명이 요청을 보내면** 어떻게 될까요?

이것이 오늘의 핵심 질문입니다.

### 1.2 동기 처리: 한 번에 하나씩

동기(Synchronous) 처리는 **작업을 순서대로, 하나씩** 수행하는 방식입니다.
앞의 작업이 끝나야 다음 작업을 시작합니다.

일상적인 비유로 설명하겠습니다.

![image.png](lms-img:0)

세 명의 고객(A, B, C)이 차례로 서 있으며, 각각 아메리카노, 라떼, 주스를 주문한 상태입니다.  
한 명의 바리스타가 주문을 하나씩 처리하고 있습니다.  
하단에 시간 표시(0분, 3분, 6분, 9분)는 고객 A는 대기 시간 0분, 고객 B는 3분, 고객 C는 6분 뒤에 음료를 받게 된다는 것을 시각적인 막대로 명확히 보여줍니다.

이것을 API 서버에 대입하면:

```
[동기 API 서버 : 모델 추론에 3초 소요]

요청 A 도착 (0초)  → 추론 시작 → 추론 완료 (3초) → 응답
요청 B 도착 (0초)  → 대기...                      → 추론 시작 (3초) → 응답 (6초)
요청 C 도착 (0초)  → 대기...                       → 대기...        → 추론 시작 (6초) → 응답 (9초)

시간축:
0초 ─────── 3초 ─────── 6초 ─────── 9초
 ███████████                              요청 A (3초 소요)
             ███████████                  요청 B (6초 대기 포함)
                          ███████████     요청 C (9초 대기 포함)
```

> 요청 B와 C는 서버가 바빠서가 아니라, **앞의 요청이 끝나기를 기다리느라** 느려진 것입니다.

### 1.3 비동기 처리: 대기 시간을 활용

비동기(Asynchronous) 처리는 **어떤 작업이 대기 상태일 때, 다른 작업을 처리**하는 방식입니다.

![image.png](lms-img:0)

직원이 A의 주문을 받고 에스프레소 머신을 가동합니다.  
머신이 돌아가는 그 시간 동안, 직원은 B의 주문을 받습니다.  
  
직원은 B의 우유 스티밍을 시작합니다 (B 스티밍 중).  
머신(A 추출 중)과 스티밍(B 스티밍 중)이 동시에 돌아갑니다.  
직원은 C의 주문을 받습니다.  

직원은 C의 블렌더를 가동합니다 (C 블렌딩 중).  
머신(A), 스티머(B), 블렌더(C) 세 기계가 동시에 작동하고, 직원은 세 작업을 오갑니다.  

이렇게 9분이 걸리던 일을 3분만에 수행해내었습니다.

> 비동기의 핵심은 **직원(CPU)을 늘리는 것이 아니라,
> 대기 시간에 놀지 않고 다른 일을 하는 것**입니다.

### 1.4 API 서버에서 "대기"란 무엇입니까?

웹 서버가 처리하는 작업은 크게 두 종류로 나뉩니다.

![image.png](lms-img:0)

여기서 중요한 점이 있습니다:

> **모델 추론은 CPU 바운드 작업입니다.**

모델이 `forward()` 연산을 수행하는 동안 CPU(또는 GPU)는 쉬지 않고 계산합니다.
이 시간에 "다른 일을 할 수 있는 대기 시간"이 존재하지 않습니다.

그렇다면 비동기가 모델 추론에는 의미가 없을까요?
결론부터 말씀드리면, **의미가 있습니다**. 하지만 단순한 `async/await`만으로는 부족하고,
추가적인 패턴이 필요합니다. 이것이 오늘 섹션 4에서 다룰 `run_in_executor`입니다.

### 1.5 FastAPI의 요청 처리 방식

FastAPI가 내부적으로 요청을 어떻게 처리하는지 이해하면,
"왜 동기 추론이 문제인지"가 명확해집니다.

```
FastAPI는 내부적으로 단일 이벤트 루프(Event Loop)에서 동작합니다.

[이벤트 루프 : 단일 스레드]

  요청 A 도착 → 핸들러 실행 → ... → 응답 반환
  요청 B 도착 → (A가 끝날 때까지 대기) → 핸들러 실행 → ...
```

FastAPI는 함수의 정의 방식에 따라 다르게 동작합니다:

```python
# 방식 1: 일반 함수 (def)
@app.post("/predict")
def predict(request: PredictRequest):
    result = model(input_tensor)    # 이 동안 이벤트 루프가 블로킹되지 않음
    return result                   # FastAPI가 별도 스레드풀에서 실행해줌

# 방식 2: 비동기 함수 (async def)
@app.post("/predict")
async def predict(request: PredictRequest):
    result = model(input_tensor)    # 주의 : 이 동안 이벤트 루프가 블로킹됨!
    return result                   # 다른 요청을 받을 수 없음
```

> **주의** : **이것이 오늘의 핵심 함정입니다.**
>
> `async def`로 선언한 함수 안에서 **동기 작업(모델 추론)**을 수행하면,
> 이벤트 루프가 그 작업이 끝날 때까지 멈춥니다.
> 멈춰 있는 동안 다른 요청은 처리되지 않습니다.
>
> 반면 일반 `def`로 선언하면, FastAPI가 자동으로 별도 스레드에서 실행합니다.
> 하지만 이 방식은 스레드 수에 제한이 있어, 최적의 해결책은 아닙니다.

이 문제를 섹션 3에서 직접 재현하고, 섹션 4에서 해결하겠습니다.

### 1.6 정리: 오늘 풀어야 할 문제

FastAPI 기초 노트에서 만든 API는 한 번에 하나의 요청만 처리할 수 있습니다.  
모델 추론이 3초 걸린다면, 10명이 동시에 요청하면 마지막 사용자는 30초를 기다려야 합니다.

해결 방향  
1. async/await의 원리를 이해한다 (섹션 2)  
2. 문제를 직접 재현한다 (섹션 3)  
3. run_in_executor로 해결한다 (섹션 4)  
4. 에러 핸들링으로 안정성을 높인다 (섹션 5)  
5. 개선 효과를 측정한다 (섹션 6)

## 2. async/await의 기본 원리

> **학습 목표**
> - `async def`와 `await`의 의미를 코드로 이해합니다.
> - 동기 함수와 비동기 함수의 실행 시간 차이를 직접 비교합니다.

### 2.1 async/await란 무엇입니까?

```python
async def   → "이 함수는 비동기 함수입니다" (중간에 멈췄다가 다시 실행될 수 있음)
await       → "이 작업이 끝날 때까지 기다리되, 그 동안 다른 일을 해도 됩니다"
```

핵심 차이는 `time.sleep()` vs `asyncio.sleep()`입니다:

```
time.sleep(3)          → CPU를 3초간 점유. 그 동안 아무것도 할 수 없음.
await asyncio.sleep(3) → "3초 후에 깨워줘"라고 등록 후, 다른 작업 처리.
```

### 2.2 실습: 동기 vs 비동기 실행 시간 비교

위에 소개한 내용을 직접 확인해보겠습니다.

In [1]:
import nest_asyncio
nest_asyncio.apply()

import time
import asyncio

In [2]:
# 동기 방식: 순차 실행
def sync_task(name, seconds):
    print(f"  [{name}] 시작")
    time.sleep(seconds)
    print(f"  [{name}] 완료 ({seconds}초)")

print("===== 동기 실행 =====")
start = time.time()
sync_task("작업A", 2)
sync_task("작업B", 2)
sync_task("작업C", 2)
print(f"\n총 소요 시간: {time.time() - start:.1f}초")

===== 동기 실행 =====
  [작업A] 시작
  [작업A] 완료 (2초)
  [작업B] 시작
  [작업B] 완료 (2초)
  [작업C] 시작
  [작업C] 완료 (2초)

총 소요 시간: 6.0초


In [3]:
# 비동기 방식: 동시 실행
async def async_task(name, seconds):
    print(f"  [{name}] 시작")
    await asyncio.sleep(seconds)
    print(f"  [{name}] 완료 ({seconds}초)")

async def run_async():
    print("===== 비동기 실행 =====")
    start = time.time()
    await asyncio.gather(
        async_task("작업A", 2),
        async_task("작업B", 2),
        async_task("작업C", 2),
    )
    print(f"\n총 소요 시간: {time.time() - start:.1f}초")

await run_async()

===== 비동기 실행 =====
  [작업A] 시작
  [작업B] 시작
  [작업C] 시작
  [작업A] 완료 (2초)
  [작업B] 완료 (2초)
  [작업C] 완료 (2초)

총 소요 시간: 2.0초


결과를 비교합니다:

```
동기: 2초 + 2초 + 2초 = 6초 (순차 실행)
비동기: max(2초, 2초, 2초) = 2초 (동시 실행)
```

> `await`를 만나면, 이벤트 루프는 대기를 등록하고 바로 다음 작업을 시작합니다.  
> 세 작업의 "시작"이 거의 동시에 출력된 것이 그 증거입니다.

위에서 논의한 내용을 짧게 정리해보겠습니다

![image.png](lms-img:0)

### 체크포인트

1. `time.sleep(3)`과 `await asyncio.sleep(3)`의 핵심 차이는 무엇입니까?
2. 모델 추론처럼 CPU를 계속 사용하는 작업에서 `async/await`만으로 동시 처리가 안 되는 이유는 무엇입니까?

## 3. 문제 시연: 동기 추론이 서버를 멈추는 순간

---

> **학습 목표**
> - 동기 추론이 서버에 미치는 영향을 실제 코드로 재현합니다.
> - 동시 요청 시 응답 시간이 어떻게 누적되는지 측정합니다.
> - `def`와 `async def`의 동작 차이를 실험으로 확인합니다.

### 3.1 실험 설계

문제를 명확하게 보여주기 위해, 의도적으로 **추론 시간이 긴 서버**를 만들겠습니다.

실제 모델 추론 대신 `time.sleep()`으로 추론 지연을 시뮬레이션합니다.
이렇게 하면 추론 시간을 정확히 통제할 수 있어 실험 결과가 깔끔합니다.

In [4]:
%%writefile app/main_sync_problem.py
"""
Day 3 - 섹션 3: 동기 추론의 문제점을 보여주는 서버
두 가지 버전의 엔드포인트를 비교합니다.
"""
import time
from fastapi import FastAPI

app = FastAPI(title="Sync vs Async Problem Demo")

INFERENCE_TIME = 3   # 추론에 3초 걸린다고 가정

# ===== 버전 1: async def 안에서 동기 작업 (문제 있음) =====
@app.post("/predict/blocking")
async def predict_blocking():
    """
    ⚠️ 문제 버전: async def 안에서 time.sleep (동기 블로킹)
    이벤트 루프가 멈추므로, 동시 요청을 처리할 수 없습니다.
    """
    time.sleep(INFERENCE_TIME)   # 동기 블로킹 — 이벤트 루프가 멈춤
    return {"result": "완료", "method": "blocking", "duration": INFERENCE_TIME}


# ===== 버전 2: 일반 def (FastAPI가 스레드풀에서 실행) =====
@app.post("/predict/threadpool")
def predict_threadpool():
    """
    일반 def: FastAPI가 자동으로 별도 스레드에서 실행합니다.
    이벤트 루프는 블로킹되지 않지만, 스레드풀 크기에 제한이 있습니다.
    """
    time.sleep(INFERENCE_TIME)
    return {"result": "완료", "method": "threadpool", "duration": INFERENCE_TIME}


# 헬스체크: 서버가 응답 가능한 상태인지 확인용
@app.get("/health")
async def health():
    return {"status": "healthy"}

Writing app/main_sync_problem.py


In [1]:
# 서버 실행
# ⚠️ 이전 섹션/Day에서 서버를 실행했다면, 반드시 커널을 재시작하세요.
#    "Address already in use" 에러 = 같은 포트에 서버가 이미 실행 중
#    해결: Kernel → Restart Kernel 후 이 셀부터 다시 실행

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_sync_problem:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ 서버 시작됨")

INFO:     Started server process [22488]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨


---

### 3.2 실험 1: 동시 요청 시 blocking 엔드포인트의 문제

3개의 요청을 **동시에** 보내고, 각각의 응답 시간을 측정합니다.

In [2]:
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

def send_request(url, request_id):
    """단일 요청을 보내고 소요 시간을 측정합니다."""
    start = time.time()
    response = requests.post(url)
    elapsed = time.time() - start
    return {
        "request_id": request_id,
        "elapsed": round(elapsed, 1),
        "status": response.status_code,
    }

def concurrent_test(url, n_requests=3):
    """n개의 요청을 동시에 보냅니다."""
    print(f"\n{'='*55}")
    print(f"  {n_requests}개 동시 요청 → {url}")
    print(f"{'='*55}")

    start = time.time()

    with ThreadPoolExecutor(max_workers=n_requests) as executor:
        futures = {
            executor.submit(send_request, url, i+1): i
            for i in range(n_requests)
        }

        results = []
        for future in as_completed(futures):
            results.append(future.result())

    total = time.time() - start

    # 결과 출력 (요청 ID순으로 정렬)
    for r in sorted(results, key=lambda x: x["request_id"]):
        print(f"  요청 #{r['request_id']}: {r['elapsed']}초")

    print(f"\n  전체 소요 시간: {round(total, 1)}초")
    return total

In [3]:
# 실험 1: blocking 엔드포인트 (async def + time.sleep)
total_blocking = concurrent_test("http://localhost:8000/predict/blocking", n_requests=3)


  3개 동시 요청 → http://localhost:8000/predict/blocking
INFO:     127.0.0.1:4042 - "POST /predict/blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:4043 - "POST /predict/blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:4044 - "POST /predict/blocking HTTP/1.1" 200 OK
  요청 #1: 11.0초
  요청 #2: 8.0초
  요청 #3: 5.0초

  전체 소요 시간: 11.0초


결과를 분석합니다:

```
추론 시간이 3초인데, 3개 동시 요청의 결과:

요청 #1: 3초  (즉시 처리)
요청 #2: 6초  (요청 #1이 끝날 때까지 3초 대기 + 추론 3초)
요청 #3: 9초  (요청 #1, #2가 끝날 때까지 6초 대기 + 추론 3초)

→ 완전한 순차 실행입니다. 동시 처리가 전혀 이루어지지 않았습니다.
```

---

### 3.3 실험 2: threadpool 엔드포인트와 비교

In [4]:
# 실험 2: threadpool 엔드포인트 (일반 def)
total_threadpool = concurrent_test("http://localhost:8000/predict/threadpool", n_requests=3)


  3개 동시 요청 → http://localhost:8000/predict/threadpool
INFO:     127.0.0.1:4054 - "POST /predict/threadpool HTTP/1.1" 200 OK
INFO:     127.0.0.1:4053 - "POST /predict/threadpool HTTP/1.1" 200 OK
INFO:     127.0.0.1:4055 - "POST /predict/threadpool HTTP/1.1" 200 OK
  요청 #1: 5.1초
  요청 #2: 5.1초
  요청 #3: 5.1초

  전체 소요 시간: 5.1초


결과를 비교합니다:

![image.png](lms-img:0)

---

### 3.4 실험 3: blocking이 헬스체크까지 막는 현상

blocking 엔드포인트의 더 심각한 문제를 확인합니다.
추론 중에 헬스체크 요청이 들어오면 어떻게 될까요?

In [5]:
import threading

def test_health_during_inference(predict_url):
    """추론 중에 헬스체크가 응답하는지 테스트합니다."""

    results = {}

    def send_predict():
        start = time.time()
        requests.post(predict_url)
        results["predict"] = round(time.time() - start, 1)

    def send_health():
        time.sleep(0.5)   # 추론이 시작된 후 0.5초 뒤에 헬스체크
        start = time.time()
        resp = requests.get("http://localhost:8000/health")
        results["health"] = round(time.time() - start, 1)

    t1 = threading.Thread(target=send_predict)
    t2 = threading.Thread(target=send_health)
    t1.start()
    t2.start()
    t1.join()
    t2.join()

    return results

In [6]:
# blocking 버전
print("===== /predict/blocking 중 헬스체크 =====")
r = test_health_during_inference("http://localhost:8000/predict/blocking")
print(f"  추론 응답: {r['predict']}초")
print(f"  헬스체크 응답: {r['health']}초    ← 단순 상태 확인인데 2.5초 대기!")

print()

# threadpool 버전
print("===== /predict/threadpool 중 헬스체크 =====")
r = test_health_during_inference("http://localhost:8000/predict/threadpool")
print(f"  추론 응답: {r['predict']}초")
print(f"  헬스체크 응답: {r['health']}초    ← 즉시 응답!")

===== /predict/blocking 중 헬스체크 =====
INFO:     127.0.0.1:4440 - "POST /predict/blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:4442 - "GET /health HTTP/1.1" 200 OK
  추론 응답: 5.0초
  헬스체크 응답: 4.5초    ← 단순 상태 확인인데 2.5초 대기!

===== /predict/threadpool 중 헬스체크 =====
INFO:     127.0.0.1:4446 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:4445 - "POST /predict/threadpool HTTP/1.1" 200 OK
  추론 응답: 5.0초
  헬스체크 응답: 2.0초    ← 즉시 응답!


> **주의** : **blocking 버전에서는 헬스체크조차 대기합니다.**
>
> 이것이 실무에서 심각한 이유:
> - 로드밸런서는 헬스체크로 서버 상태를 판단합니다.
> - 헬스체크가 응답하지 않으면 "서버 다운"으로 판단하여 트래픽을 차단합니다.
> - 실제로는 서버가 살아있는데, 추론 때문에 응답을 못 하는 것뿐입니다.
> - 결과적으로 정상 서버가 풀에서 빠지는 심각한 문제가 발생합니다.

---

### 3.5 문제 원인 정리

```
[async def + 동기 작업의 문제]

async def predict_blocking():
    time.sleep(3)    ← 이벤트 루프 스레드에서 직접 실행
                        이벤트 루프가 3초간 완전히 멈춤
                        그 동안 다른 요청 처리 불가
                        헬스체크도 불가

[일반 def의 동작]

def predict_threadpool():
    time.sleep(3)    ← FastAPI가 별도 스레드풀(ThreadPoolExecutor)에서 실행
                        이벤트 루프는 자유로움
                        다른 요청 처리 가능
                        헬스체크 즉시 응답 가능
```

> **그러면 일반 def만 쓰면 되지 않습니까?**
>
> 일반 `def`를 사용하면 FastAPI가 기본 스레드풀에서 실행해줍니다.
> 이것만으로도 대부분의 경우 충분히 동작합니다.
>
> 하지만 두 가지 한계가 있습니다:
> 1. 기본 스레드풀 크기가 제한적입니다 (보통 40개).
> 2. 스레드풀 크기, 실행 방식을 직접 제어할 수 없습니다.
>
> 다음 섹션에서 배울 `run_in_executor` 패턴을 사용하면,
> 스레드풀을 명시적으로 제어하면서도 `async def`의 장점을 모두 활용할 수 있습니다.

---

### 체크포인트

1. `async def` 안에서 `time.sleep(3)`을 호출하면 왜 다른 요청까지 지연됩니까?
2. 일반 `def`로 선언된 엔드포인트는 FastAPI가 내부적으로 어떻게 처리합니까?
3. blocking 추론 중 헬스체크까지 막히면 실무에서 어떤 문제가 발생할 수 있습니까?

---

> **다음 섹션에서는** `run_in_executor` 패턴으로 이 문제를 해결합니다.
> `async def`의 장점을 유지하면서, 동기 작업도 이벤트 루프를 막지 않도록 만듭니다.

## 4. 해결 패턴: run_in_executor로 블로킹 방지하기

---

> **학습 목표**
> - `run_in_executor`의 동작 원리를 이해합니다.
> - 모델 추론을 이벤트 루프를 막지 않으면서 실행하는 패턴을 익힙니다.
> - ThreadPoolExecutor의 크기를 조절하는 방법을 알 수 있습니다.
> - FastAPI 기초 노트에서 만든 API에 이 패턴을 적용할 수 있습니다.

### 4.1 run_in_executor란 무엇입니까?

`run_in_executor`는 **동기 함수를 별도 스레드(또는 프로세스)에서 실행**하고,
그 결과를 비동기적으로 기다리는 패턴입니다.

![image.png](lms-img:0)

코드로 보면 단 한 줄입니다:

```python
import asyncio

# Before: 이벤트 루프를 막는 방식
async def predict_blocking():
    result = model(input_tensor)     # 이벤트 루프 멈춤
    return result

# After: 이벤트 루프를 막지 않는 방식
async def predict_non_blocking():
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        None,                        # None = 기본 스레드풀 사용
        model,                       # 실행할 함수
        input_tensor,                # 함수에 전달할 인자
    )
    return result
```

> `await loop.run_in_executor(None, func, arg)`
>
> 이 한 줄이 하는 일:
> 1. `func(arg)`를 별도 스레드에서 실행합니다.
> 2. 이벤트 루프는 그 동안 다른 작업을 처리합니다.
> 3. `func`이 완료되면 결과를 반환합니다.

### 4.2 실습: 세 가지 버전 비교

섹션 3의 실험에 `run_in_executor` 버전을 추가하여 세 가지를 비교합니다.

In [7]:
%%writefile app/main_async_solution.py
"""
Day 3 - 섹션 4: 세 가지 동시 처리 방식 비교
"""
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI

app = FastAPI(title="Async Solution Demo")

INFERENCE_TIME = 3

# 커스텀 스레드풀 생성 (최대 4개 스레드)
inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="inference")


def heavy_inference():
    """동기 함수: 모델 추론을 시뮬레이션합니다."""
    time.sleep(INFERENCE_TIME)
    return {"result": "완료", "duration": INFERENCE_TIME}


# ===== 버전 1: async def + 동기 작업 (문제 있음) =====
@app.post("/predict/v1-blocking")
async def predict_v1():
    """❌ 이벤트 루프를 막습니다."""
    time.sleep(INFERENCE_TIME)
    return {"method": "v1-blocking", "duration": INFERENCE_TIME}


# ===== 버전 2: 일반 def (FastAPI 자동 스레드풀) =====
@app.post("/predict/v2-def")
def predict_v2():
    """⭕ FastAPI가 자동으로 별도 스레드에서 실행합니다."""
    time.sleep(INFERENCE_TIME)
    return {"method": "v2-def", "duration": INFERENCE_TIME}


# ===== 버전 3: async def + run_in_executor (권장) =====
@app.post("/predict/v3-executor")
async def predict_v3():
    """✅ 명시적으로 스레드풀에 위임합니다."""
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        inference_executor,    # 커스텀 스레드풀 사용
        heavy_inference,       # 실행할 동기 함수
    )
    return {"method": "v3-executor", **result}


@app.get("/health")
async def health():
    return {"status": "healthy"}

Writing app/main_async_solution.py


In [1]:
# ⚠️ 섹션 3에서 서버를 실행했다면, 반드시 커널을 재시작하세요.

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_async_solution:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ 서버 시작됨")

INFO:     Started server process [6632]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ 서버 시작됨


In [2]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

def concurrent_test(url, n_requests=3):
    """n개의 요청을 동시에 보내고 결과를 측정합니다."""
    def send(i):
        start = time.time()
        resp = requests.post(url)
        return {"id": i+1, "elapsed": round(time.time() - start, 1)}

    start = time.time()
    with ThreadPoolExecutor(max_workers=n_requests) as ex:
        futures = [ex.submit(send, i) for i in range(n_requests)]
        results = [f.result() for f in as_completed(futures)]
    total = round(time.time() - start, 1)

    for r in sorted(results, key=lambda x: x["id"]):
        print(f"  요청 #{r['id']}: {r['elapsed']}초")
    print(f"  전체: {total}초\n")
    return total

In [3]:
print("=" * 50)
print("버전 1: async def + time.sleep (blocking)")
print("=" * 50)
t1 = concurrent_test("http://localhost:8000/predict/v1-blocking")

버전 1: async def + time.sleep (blocking)
INFO:     127.0.0.1:10887 - "POST /predict/v1-blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:10888 - "POST /predict/v1-blocking HTTP/1.1" 200 OK
INFO:     127.0.0.1:10889 - "POST /predict/v1-blocking HTTP/1.1" 200 OK
  요청 #1: 8.0초
  요청 #2: 11.0초
  요청 #3: 5.0초
  전체: 11.0초



In [4]:
print("=" * 50)
print("버전 2: 일반 def (FastAPI 자동 스레드풀)")
print("=" * 50)
t2 = concurrent_test("http://localhost:8000/predict/v2-def")

버전 2: 일반 def (FastAPI 자동 스레드풀)
INFO:     127.0.0.1:10894 - "POST /predict/v2-def HTTP/1.1" 200 OK
INFO:     127.0.0.1:10895 - "POST /predict/v2-def HTTP/1.1" 200 OK
INFO:     127.0.0.1:10896 - "POST /predict/v2-def HTTP/1.1" 200 OK
  요청 #1: 5.1초
  요청 #2: 5.1초
  요청 #3: 5.1초
  전체: 5.1초



In [6]:
print("=" * 50)
print("버전 3: async def + run_in_executor (권장)")
print("=" * 50)
t3 = concurrent_test("http://localhost:8000/predict/v3-executor")

버전 3: async def + run_in_executor (권장)
INFO:     127.0.0.1:7292 - "POST /predict/v3-executor HTTP/1.1" 200 OK
INFO:     127.0.0.1:7293 - "POST /predict/v3-executor HTTP/1.1" 200 OK
INFO:     127.0.0.1:7294 - "POST /predict/v3-executor HTTP/1.1" 200 OK
  요청 #1: 5.0초
  요청 #2: 5.0초
  요청 #3: 5.0초
  전체: 5.0초



### 4.3 버전 2와 버전 3의 차이

결과만 보면 동일하지만, `run_in_executor`를 쓰는 핵심 이유가 있습니다:

```
일반 def (버전 2):
  → FastAPI 기본 스레드풀을 사용합니다.
  → 스레드풀 크기를 제어할 수 없습니다.
  → 다른 엔드포인트와 스레드풀을 공유합니다.

async def + run_in_executor (버전 3):
  → 추론 전용 스레드풀을 직접 만들어 사용합니다.
  → 크기를 제어할 수 있습니다 (max_workers=4).
  → 추론 부하가 다른 엔드포인트에 영향을 주지 않습니다.
```

> 간단한 프로젝트에서는 일반 `def`로 충분하지만,
> 추론 전용 스레드풀을 분리하고 싶다면 `run_in_executor`를 사용합니다.
> 이 과정에서는 `run_in_executor` 패턴을 기본으로 사용합니다.

### 4.4 FastAPI 기초 노트의 API에 적용하기

FastAPI 기초 노트에서 만든 실제 추론 API(`app/main.py`)에 `run_in_executor`를 적용합니다.

In [7]:
%%writefile app/main_v2.py
"""
Day 3 - Day 2 API에 비동기 패턴 적용
app/main.py의 개선 버전입니다.
"""
import io
import base64
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException

from app.schemas import (
    PixelPredictRequest,
    ImagePredictRequest,
    PredictResponse,
)
from app.model_utils import load_model, predict, preprocess


# ===== 앱 생성 =====
app = FastAPI(
    title="MNIST Prediction API (Async)",
    description="비동기 처리가 적용된 MNIST 추론 API",
    version="2.0.0",
)

# ===== 추론 전용 스레드풀 =====
inference_executor = ThreadPoolExecutor(
    max_workers=4,
    thread_name_prefix="inference",
)

# ===== 모델 로드 =====
MODEL_PATH = "models/mnist_state_dict.pth"
model = load_model(MODEL_PATH)


# ===== 동기 추론 함수 (스레드풀에서 실행될 함수) =====
def run_inference(image_tensor: torch.Tensor) -> dict:
    """모델 추론을 수행합니다. 이 함수는 별도 스레드에서 실행됩니다."""
    return predict(model, image_tensor)


# ===== 엔드포인트 =====

@app.get("/health", tags=["System"])
async def health_check():
    return {"status": "healthy", "model_loaded": model is not None}


@app.post("/predict/pixels", response_model=PredictResponse, tags=["Inference"])
async def predict_from_pixels(request: PixelPredictRequest):
    """비동기 버전: 픽셀 배열로 추론"""
    try:
        # 전처리 (가벼운 작업 — 이벤트 루프에서 직접 수행)
        pixel_array = np.array(request.pixels, dtype=np.float32)
        pixel_tensor = torch.from_numpy(pixel_array)
        pixel_tensor = (pixel_tensor - 0.1307) / 0.3081
        pixel_tensor = pixel_tensor.unsqueeze(0).unsqueeze(0)

        # 추론 (무거운 작업 — 별도 스레드에서 실행)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            inference_executor,
            run_inference,
            pixel_tensor,
        )

        return PredictResponse(
            success=True,
            predicted_class=result["predicted_class"],
            confidence=result["confidence"],
            probabilities=result["probabilities"] if request.return_probabilities else None,
        )

    except Exception as e:
        raise HTTPException(status_code=400, detail=f"추론 실패: {str(e)}")


@app.post("/predict/image", response_model=PredictResponse, tags=["Inference"])
async def predict_from_image(request: ImagePredictRequest):
    """비동기 버전: Base64 이미지로 추론"""
    try:
        # 전처리 (가벼운 작업)
        image_bytes = base64.b64decode(request.image_base64)
        image = Image.open(io.BytesIO(image_bytes))
        image_tensor = preprocess(image).unsqueeze(0)

        # 추론 (무거운 작업 — 별도 스레드에서 실행)
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            inference_executor,
            run_inference,
            image_tensor,
        )

        return PredictResponse(
            success=True,
            predicted_class=result["predicted_class"],
            confidence=result["confidence"],
            probabilities=result["probabilities"] if request.return_probabilities else None,
        )

    except base64.binascii.Error:
        raise HTTPException(status_code=400, detail="유효하지 않은 Base64 문자열입니다.")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"이미지 처리 실패: {str(e)}")

Writing app/main_v2.py


변경된 부분을 정리합니다:

```
FastAPI 기초 노트 (app/main.py)        4장 (app/main_v2.py)
──────────────────                     ──────────────────────

def predict_from_pixels(...):          async def predict_from_pixels(...):
    ...                                    ...
    result = predict(model, tensor)        result = await loop.run_in_executor(
    ...                                        inference_executor,
                                               run_inference,
                                               tensor,
                                           )
                                           ...

변경: 2줄                               추가: inference_executor, run_inference
```

> 핵심 변경은 **추론 호출 한 줄**뿐입니다.
> 나머지 전처리, 검증, 에러 처리 코드는 그대로 유지됩니다.

### 4.5 스레드풀 크기 가이드

```
CPU 추론 (GPU 없음):
  - CPU 코어 수와 비슷하게 설정합니다.
  - 예: 4코어 → max_workers=4
  - 너무 많으면 컨텍스트 스위칭 오버헤드가 발생합니다.

GPU 추론:
  - GPU 메모리 제한이 있으므로 1~2로 설정합니다.
  - GPU는 내부적으로 병렬 처리하므로, 스레드를 늘려도 속도 향상이 크지 않습니다.
```

In [8]:
import os

# CPU 코어 수 확인
cpu_count = os.cpu_count()
print(f"CPU 코어 수: {cpu_count}")
print(f"권장 max_workers (CPU 추론): {cpu_count}")
print(f"권장 max_workers (GPU 추론): 1~2")

CPU 코어 수: 12
권장 max_workers (CPU 추론): 12
권장 max_workers (GPU 추론): 1~2


### 4.6 정리: 어떤 패턴을 선택해야 합니까?

![image.png](lms-img:0)

> 이 과정의 프로젝트에서는 `run_in_executor` 패턴을 사용합니다.
> 한번 익혀두면 실무에서도 그대로 적용할 수 있기 때문입니다.

---

### 체크포인트

1. `run_in_executor`가 이벤트 루프 블로킹을 방지하는 원리는 무엇입니까?
2. `run_in_executor`의 첫 번째 인자에 `None`을 넣으면 어떤 스레드풀이 사용됩니까?
3. 일반 `def`와 `async def + run_in_executor`의 핵심 차이는 무엇입니까?
4. GPU 추론 시 스레드풀 크기를 1\~2로 제한하는 이유는 무엇입니까?

---

> **다음 섹션에서는** 서버의 안정성을 높이기 위한 에러 핸들링과 로깅을 다룹니다.

## 5. 에러 핸들링과 로깅

---

> **학습 목표**
> - 예외 발생 시 서버가 죽지 않고, 안전한 에러 응답을 반환하는 구조를 만들 수 있습니다.
> - Python logging 모듈을 사용하여 로그를 남길 수 있습니다.

### 5.1 왜 에러 핸들링이 중요합니까?

입력이 올바르더라도 **서버 내부에서 에러가 발생**할 수 있습니다.

```
- 모델 파일이 손상되어 추론 중 에러 발생
- GPU 메모리 부족 (OOM)
- 이미지 디코딩 실패 (손상된 파일)
```

에러를 처리하지 않으면:

```
→ 스택 트레이스가 클라이언트에 노출 (보안 위험)
→ 에러 원인 추적 불가 (로그가 없음)

에러 핸들링 적용 후:
→ 서버 로그에 상세 정보 기록
→ 클라이언트에는 안전한 메시지만 반환
→ 서버는 계속 동작
```

### 5.2 글로벌 Exception Handler

매 엔드포인트마다 try/except를 반복하는 대신,
**글로벌 핸들러 하나**로 모든 예외를 중앙에서 처리합니다.

In [9]:
%%writefile app/error_handlers.py
"""
Day 3 - 글로벌 에러 핸들러
"""
import traceback
import logging

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")


def register_error_handlers(app: FastAPI):
    """FastAPI 앱에 글로벌 에러 핸들러를 등록합니다."""

    @app.exception_handler(Exception)
    async def general_error_handler(request: Request, exc: Exception):
        """모든 예외를 잡아서 안전한 응답을 반환합니다."""
        logger.error(
            f"에러 발생: {type(exc).__name__}: {exc}\n"
            f"경로: {request.method} {request.url}\n"
            f"스택 트레이스:\n{traceback.format_exc()}"
        )
        return JSONResponse(
            status_code=500,
            content={
                "success": False,
                "error": "서버 내부 오류가 발생했습니다.",
            }
            # ⚠️ 클라이언트에게는 상세 정보를 노출하지 않습니다.
            # 상세 정보는 서버 로그에만 기록됩니다.
        )

Writing app/error_handlers.py


> 핵심 원칙:
> - **로그에는** 스택 트레이스, 요청 경로, 원본 에러 등 상세 정보를 기록합니다.
> - **클라이언트에게는** "서버 내부 오류가 발생했습니다"만 반환합니다.
> - 내부 코드 경로, 라이브러리 버전 등이 노출되면 보안 위험이 됩니다.

> **더 알고 싶다면**
>
> 실무에서는 커스텀 예외 클래스(`InferenceError`, `PreprocessError` 등)를 정의하여
> 에러 유형별로 다른 상태 코드를 반환하기도 합니다.
> 이 과정에서는 범용 핸들러 하나로 충분합니다.

### 5.3 Python logging 설정

`print()` 대신 `logging` 모듈을 사용해야 하는 이유:

```
print("에러 발생")           → 시간 정보 없음, 심각도 구분 없음, 파일 저장 불가

logger.error("에러 발생")    → 2024-01-15 14:30:22 ERROR [ml_api] 에러 발생
                               시간, 심각도, 모듈명이 자동 포함
```

In [10]:
%%writefile app/logger_config.py
"""
Day 3 - 로깅 설정
"""
import logging
import sys


def setup_logger(name: str = "ml_api", level: str = "INFO") -> logging.Logger:
    """콘솔 로거를 설정합니다."""
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))

    if logger.handlers:
        return logger

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.DEBUG)

    formatter = logging.Formatter(
        fmt="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

    return logger

Writing app/logger_config.py


> 실무에서는 파일 핸들러나 외부 로그 서비스(CloudWatch, Datadog 등)로
> 로그를 보내지만, 이 과정에서는 콘솔 로그로 충분합니다.

In [11]:
# 로거 테스트
from app.logger_config import setup_logger

logger = setup_logger("ml_api")

logger.info("서버가 시작되었습니다.")
logger.warning("GPU 메모리가 80%를 초과했습니다.")
logger.error("모델 추론 중 에러가 발생했습니다.")

2026-10-29 12:53:03 INFO     [ml_api] 서버가 시작되었습니다.
2026-10-29 12:53:03 WARNING  [ml_api] GPU 메모리가 80%를 초과했습니다.
2026-10-29 12:53:03 ERROR    [ml_api] 모델 추론 중 에러가 발생했습니다.


```
2024-01-15 14:30:22 INFO     [ml_api] 서버가 시작되었습니다.
2024-01-15 14:30:22 WARNING  [ml_api] GPU 메모리가 80%를 초과했습니다.
2024-01-15 14:30:22 ERROR    [ml_api] 모델 추론 중 에러가 발생했습니다.
```

로그 레벨의 의미:

```
INFO     → 정상 동작 기록 (서버 시작, 모델 로드 등)
WARNING  → 주의가 필요한 상황 (메모리 부족 임박 등)
ERROR    → 에러 발생 (추론 실패 등)
CRITICAL → 심각한 에러 (서버 다운 위험)
```

### 5.4 요청/응답 로깅 미들웨어

모든 요청과 응답을 자동으로 로깅하는 미들웨어입니다.
아래 코드를 그대로 사용하시면 됩니다.

In [12]:
%%writefile app/middleware.py
"""
Day 3 - 요청/응답 로깅 미들웨어
모든 요청의 메서드, 경로, 응답 시간, 상태 코드를 자동 로깅합니다.
"""
import time
import logging
from fastapi import Request
from starlette.middleware.base import BaseHTTPMiddleware

logger = logging.getLogger("ml_api")


class RequestLoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        start_time = time.time()
        response = await call_next(request)
        duration = round(time.time() - start_time, 3)

        log_message = (
            f"{request.method} {request.url.path} "
            f"→ {response.status_code} "
            f"({duration}s)"
        )

        if response.status_code >= 500:
            logger.error(log_message)
        elif response.status_code >= 400:
            logger.warning(log_message)
        else:
            logger.info(log_message)

        response.headers["X-Process-Time"] = str(duration)
        return response

Writing app/middleware.py


이 미들웨어를 등록하면 서버 로그가 이렇게 보입니다:

```
2024-01-15 14:30:22 INFO     [ml_api] GET /health → 200 (0.001s)
2024-01-15 14:30:23 INFO     [ml_api] POST /predict/pixels → 200 (0.045s)
2024-01-15 14:30:24 WARNING  [ml_api] POST /predict/pixels → 422 (0.002s)
```

> 미들웨어는 `app.add_middleware(RequestLoggingMiddleware)`로 등록합니다.
> 섹션 6의 최종 서버 코드에서 실제로 적용합니다.

---

### 체크포인트

1. 글로벌 Exception Handler를 사용하면 어떤 반복을 줄일 수 있습니까?
2. 클라이언트에게 스택 트레이스를 노출하면 안 되는 이유는 무엇입니까?
3. `logging` 모듈이 `print()`보다 나은 점은 무엇입니까?

---

> **다음 섹션에서는** 오늘 배운 모든 내용을 종합하여,
> 동시 요청 테스트로 개선 효과를 직접 측정합니다.

## 6. 실습: 최종 서버 + 동시 요청 테스트

---

> **실습 목표**
> - 비동기 + 에러 핸들링 + 로깅이 적용된 최종 서버를 실행합니다.
> - 동시 요청 테스트로 동작을 확인합니다.
> - 에러 핸들링이 정상 동작하는지 확인합니다.

### 6.0 사전 준비: 의존 파일 확인  

이 섹션은 REST API 노트부터 비동기 처리 노트까지 생성한 파일들을 사용합니다.
아래 셀을 실행하면 필요한 파일이 있는지 확인하고, 없으면 자동으로 생성합니다.

> **주의** : REST API 노트부터 비동기 처리 노트까지 순서대로 진행한 분은 이 셀을 건너뛰어도 됩니다.
> 이 섹션부터 바로 시작하는 분만 실행하세요.

In [3]:
import os

# 필요한 폴더 생성
for d in ["app", "models", "data"]:
    os.makedirs(d, exist_ok=True)

# 1. app/model_utils.py (Day 1)
if not os.path.exists("app/model_utils.py"):
    print("⚠️ app/model_utils.py 없음 → 생성합니다.")
    with open("app/model_utils.py", "w", encoding="utf-8") as f:
        f.write('''import torch
import torch.nn as nn
from torchvision import transforms

class SimpleClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

preprocess = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

CLASS_NAMES = [str(i) for i in range(10)]

def load_model(model_path, num_classes=10):
    model = SimpleClassifier(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()
    return model

def predict(model, image_tensor):
    with torch.no_grad():
        output = model(image_tensor)
        probs = torch.softmax(output, dim=1)[0]
        idx = probs.argmax().item()
        return {
            "predicted_class": CLASS_NAMES[idx],
            "confidence": round(probs[idx].item(), 4),
            "probabilities": {CLASS_NAMES[i]: round(probs[i].item(), 4) for i in range(len(CLASS_NAMES))},
        }
''')
    print("  ✅ app/model_utils.py 생성 완료")
else:
    print("✅ app/model_utils.py 있음")

# 2. app/schemas.py (Day 2)
if not os.path.exists("app/schemas.py"):
    print("⚠️ app/schemas.py 없음 → 생성합니다.")
    with open("app/schemas.py", "w", encoding="utf-8") as f:
        f.write('''from pydantic import BaseModel, Field, field_validator
from typing import Optional

class PixelPredictRequest(BaseModel):
    pixels: list[list[float]] = Field(..., description="28x28 픽셀 배열")
    return_probabilities: bool = Field(default=False)
    @field_validator("pixels")
    @classmethod
    def validate_pixels(cls, v):
        if len(v) != 28:
            raise ValueError(f"28행이어야 합니다. 현재: {len(v)}행")
        for i, row in enumerate(v):
            if len(row) != 28:
                raise ValueError(f"각 행은 28열이어야 합니다. {i}번째 행: {len(row)}열")
        return v

class ImagePredictRequest(BaseModel):
    image_base64: str = Field(..., min_length=1)
    return_probabilities: bool = Field(default=False)

class PredictResponse(BaseModel):
    success: bool = Field(description="성공 여부")
    predicted_class: str = Field(description="예측 숫자 (0~9)")
    confidence: float = Field(description="확신도", ge=0.0, le=1.0)
    probabilities: Optional[dict[str, float]] = Field(default=None)
''')
    print("  ✅ app/schemas.py 생성 완료")
else:
    print("✅ app/schemas.py 있음")

# 3. app/error_handlers.py (Day 3 섹션 5)
if not os.path.exists("app/error_handlers.py"):
    print("⚠️ app/error_handlers.py 없음 → 생성합니다.")
    with open("app/error_handlers.py", "w", encoding="utf-8") as f:
        f.write('''import traceback, logging
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

logger = logging.getLogger("ml_api")

def register_error_handlers(app: FastAPI):
    @app.exception_handler(Exception)
    async def general_error_handler(request: Request, exc: Exception):
        logger.error(f"에러: {type(exc).__name__}: {exc}\\n경로: {request.method} {request.url}\\n{traceback.format_exc()}")
        return JSONResponse(status_code=500, content={"success": False, "error": "서버 내부 오류가 발생했습니다."})
''')
    print("  ✅ app/error_handlers.py 생성 완료")
else:
    print("✅ app/error_handlers.py 있음")

# 4. app/logger_config.py (Day 3 섹션 5)
if not os.path.exists("app/logger_config.py"):
    print("⚠️ app/logger_config.py 없음 → 생성합니다.")
    with open("app/logger_config.py", "w", encoding="utf-8") as f:
        f.write('''import logging, sys

def setup_logger(name="ml_api", level="INFO"):
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level))
    if logger.handlers:
        return logger
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)-8s [%(name)s] %(message)s", "%Y-%m-%d %H:%M:%S"))
    logger.addHandler(handler)
    return logger
''')
    print("  ✅ app/logger_config.py 생성 완료")
else:
    print("✅ app/logger_config.py 있음")

# 5. app/middleware.py (Day 3 섹션 5)
if not os.path.exists("app/middleware.py"):
    print("⚠️ app/middleware.py 없음 → 생성합니다.")
    with open("app/middleware.py", "w", encoding="utf-8") as f:
        f.write('''import time, logging
from fastapi import Request
from starlette.middleware.base import BaseHTTPMiddleware

logger = logging.getLogger("ml_api")

class RequestLoggingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        start = time.time()
        response = await call_next(request)
        duration = round(time.time() - start, 3)
        msg = f"{request.method} {request.url.path} -> {response.status_code} ({duration}s)"
        if response.status_code >= 500: logger.error(msg)
        elif response.status_code >= 400: logger.warning(msg)
        else: logger.info(msg)
        response.headers["X-Process-Time"] = str(duration)
        return response
''')
    print("  ✅ app/middleware.py 생성 완료")
else:
    print("✅ app/middleware.py 있음")

# 6. 모델 파일 (Day 1)
if not os.path.exists("models/mnist_state_dict.pth"):
    print("⚠️ 모델 파일 없음 → MNIST 모델을 학습하여 생성합니다. (약 1~2분 소요)")
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader
    from app.model_utils import SimpleClassifier

    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_data = DataLoader(datasets.MNIST("data", train=True, download=True, transform=transform), batch_size=64, shuffle=True)

    model = SimpleClassifier(10)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(2):
        for batch_idx, (images, labels) in enumerate(train_data):
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            if (batch_idx+1) % 300 == 0:
                print(f"  Epoch {epoch+1}, Batch {batch_idx+1}/{len(train_data)}, Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), "models/mnist_state_dict.pth")
    print("  ✅ models/mnist_state_dict.pth 생성 완료")
else:
    print("✅ models/mnist_state_dict.pth 있음")

print("\n🎉 모든 의존 파일이 준비되었습니다. 다음 셀로 진행하세요.")

⚠️ app/model_utils.py 없음 → 생성합니다.
  ✅ app/model_utils.py 생성 완료
⚠️ app/schemas.py 없음 → 생성합니다.
  ✅ app/schemas.py 생성 완료
✅ app/error_handlers.py 있음
✅ app/logger_config.py 있음
✅ app/middleware.py 있음
✅ models/mnist_state_dict.pth 있음

🎉 모든 의존 파일이 준비되었습니다. 다음 셀로 진행하세요.


🔎 **더 알아보기**
FastAPI 기초 노트를 마친 폴더에는 그 노트의 `app/schemas.py`와 `app/model_utils.py`가 있어서 6.0절 셀이 두 파일 모두 "있음"으로 넘어갑니다. 그 버전의 `app/schemas.py`에는 6.1절이 쓰는 `PixelPredictRequest`가 없으므로 6.2절 서버는 `ImportError: cannot import name 'PixelPredictRequest' from 'app.schemas'`를 내고 뜨지 못합니다. 6.2절 서버 셀은 그래도 "서버 시작됨"을 찍으므로 셀 출력에 이 오류가 있는지와 헬스체크(`/health`)가 답하는지를 먼저 확인하십시오.

이 경우 두 파일의 이름을 바꾼 뒤 6.0절 셀을 다시 실행하십시오. 그 버전의 `predict()`도 `predicted_class` 대신 `label`을 돌려주므로 `app/model_utils.py`까지 바꿔야 합니다. 6.0절 셀이 6장에 맞는 두 파일을 새로 만듭니다.

```python
import os
os.rename("app/schemas.py", "app/schemas_basic.py")
os.rename("app/model_utils.py", "app/model_utils_basic.py")
```

그다음 커널을 재시작한 뒤 6.2절 서버 셀부터 실행하십시오(코랩은 **코랩에서 할 때** 셀부터). 재시작하지 않으면 예전 `app/schemas.py`가 메모리에 남아 같은 오류가 다시 납니다.

In [2]:
import os
os.rename("app/schemas.py", "app/schemas_basic.py")
os.rename("app/model_utils.py", "app/model_utils_basic.py")

### 6.1 최종 서버 코드 통합

FastAPI 기초 노트의 API에 오늘 배운 비동기 처리, 에러 핸들링, 로깅을 모두 적용한 최종 버전입니다.

In [4]:
%%writefile app/main_final.py
"""
Day 3 최종 버전 - 비동기 + 에러 핸들링 + 로깅
"""
import io
import base64
import asyncio
from concurrent.futures import ThreadPoolExecutor

import torch
import numpy as np
from PIL import Image
from fastapi import FastAPI, HTTPException

from app.schemas import PixelPredictRequest, ImagePredictRequest, PredictResponse
from app.model_utils import load_model, predict, preprocess
from app.logger_config import setup_logger
from app.error_handlers import register_error_handlers
from app.middleware import RequestLoggingMiddleware


logger = setup_logger("ml_api")

app = FastAPI(
    title="MNIST Prediction API",
    description="비동기 처리, 에러 핸들링, 로깅이 적용된 MNIST 추론 API",
    version="3.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="inference")

MODEL_PATH = "models/mnist_state_dict.pth"
model = None


@app.on_event("startup")
async def startup():
    global model
    logger.info(f"모델 로드 중: {MODEL_PATH}")
    model = load_model(MODEL_PATH)
    logger.info("모델 로드 완료")


def run_inference(image_tensor: torch.Tensor) -> dict:
    """별도 스레드에서 실행되는 추론 함수"""
    if model is None:
        raise RuntimeError("모델이 로드되지 않았습니다")
    return predict(model, image_tensor)


@app.get("/health", tags=["System"])
async def health_check():
    return {
        "status": "healthy" if model is not None else "loading",
        "model_loaded": model is not None,
    }


@app.get("/model/info", tags=["System"])
async def model_info():
    from app.model_utils import CLASS_NAMES
    total_params = sum(p.numel() for p in model.parameters())
    return {
        "model_name": "SimpleClassifier",
        "model_path": MODEL_PATH,
        "num_classes": len(CLASS_NAMES),
        "classes": CLASS_NAMES,
        "total_parameters": total_params,
    }


@app.post("/predict/pixels", response_model=PredictResponse, tags=["Inference"])
async def predict_from_pixels(request: PixelPredictRequest):
    try:
        pixel_array = np.array(request.pixels, dtype=np.float32)
        pixel_tensor = torch.from_numpy(pixel_array)
        pixel_tensor = (pixel_tensor - 0.1307) / 0.3081
        pixel_tensor = pixel_tensor.unsqueeze(0).unsqueeze(0)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"전처리 실패: {str(e)}")

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(inference_executor, run_inference, pixel_tensor)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return PredictResponse(
        success=True,
        predicted_class=result["predicted_class"],
        confidence=result["confidence"],
        probabilities=result["probabilities"] if request.return_probabilities else None,
    )


@app.post("/predict/image", response_model=PredictResponse, tags=["Inference"])
async def predict_from_image(request: ImagePredictRequest):
    try:
        image_bytes = base64.b64decode(request.image_base64)
        image = Image.open(io.BytesIO(image_bytes))
        image_tensor = preprocess(image).unsqueeze(0)
    except base64.binascii.Error:
        raise HTTPException(status_code=400, detail="유효하지 않은 Base64 문자열입니다.")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"이미지 처리 실패: {str(e)}")

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(inference_executor, run_inference, image_tensor)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"추론 실패: {str(e)}")

    return PredictResponse(
        success=True,
        predicted_class=result["predicted_class"],
        confidence=result["confidence"],
        probabilities=result["probabilities"] if request.return_probabilities else None,
    )

Overwriting app/main_final.py


> FastAPI 기초 노트의 `app/main.py`와 비교했을 때 변경된 부분:
> - `async def` + `run_in_executor`로 추론 비동기화
> - `setup_logger`로 구조화된 로깅
> - `register_error_handlers`로 글로벌 에러 처리
> - `RequestLoggingMiddleware`로 모든 요청 자동 로깅

---

### 6.2 서버 실행 및 테스트

In [1]:
# ⚠️ 이전 섹션에서 서버를 실행했다면, 반드시 커널을 재시작하세요.
#    "Address already in use" → Kernel → Restart Kernel 후 이 셀부터 실행

# 이 노트의 일부는 Day 2 sextion 5에서 생성한 파일을 활용합니다

import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_final:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨")

✅ 서버 시작됨


🔎 **더 알아보기**
이 셀은 서버가 실제로 떴는지와 상관없이 3초 뒤에 "서버 시작됨"을 찍습니다. torch를 처음 불러오고 모델을 올리는 데 3초가 넘게 걸리면 `concurrent_pixel_test`를 부르는 셀이 연결 거부(`ConnectionError`)로 멈춥니다. 그때는 몇 초 기다렸다가 그 셀을 다시 실행하십시오.

다시 실행해도 연결 거부가 나면 서버 스레드가 이미 멈춘 상태입니다. 서버 셀 출력에 `ImportError`가 있으면 6.0절 뒤의 🔎 메모대로 두 파일 이름을 바꾸고 6.0절 셀, 커널 재시작, **코랩에서 할 때** 셀, 6.2절 서버 셀 순서로 다시 실행하십시오.

6.1절의 `@app.on_event("startup")`은 지금 버전(FastAPI 0.141.1)에서 폐기 예정 경고(`DeprecationWarning`)를 냅니다. 노트북 기본 설정에서는 이 경고가 보이지 않고 서버도 그대로 동작합니다.

새로 쓰는 코드는 lifespan으로 씁니다. 7.2절의 서버가 그 모양입니다.

#### 동시 요청 테스트

섹션 3에서 사용한 `concurrent_test()` 함수를 재사용하여 최종 서버를 테스트합니다.

In [2]:
import requests
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from torchvision import datasets

test_dataset = datasets.MNIST(root="data", train=False, download=True)

def concurrent_pixel_test(n_requests=3):
    """실제 모델 추론으로 동시 요청을 테스트합니다."""
    def send(i):
        image, label = test_dataset[i % len(test_dataset)]
        pixels = (np.array(image) / 255.0).tolist()
        start = time.time()
        resp = requests.post(
            "http://localhost:8000/predict/pixels",
            json={"pixels": pixels},
            timeout=30,
        )
        return {
            "id": i + 1,
            "elapsed": round(time.time() - start, 2),
            "status": resp.status_code,
        }

    print(f"\n{'='*50}")
    print(f"  {n_requests}개 동시 요청 (실제 추론)")
    print(f"{'='*50}")

    start = time.time()
    with ThreadPoolExecutor(max_workers=n_requests) as ex:
        futures = [ex.submit(send, i) for i in range(n_requests)]
        results = [f.result() for f in as_completed(futures)]
    total = round(time.time() - start, 2)

    for r in sorted(results, key=lambda x: x["id"]):
        print(f"  요청 #{r['id']}: {r['elapsed']}초 (HTTP {r['status']})")
    print(f"  전체: {total}초")

In [3]:
# 동시 요청 수를 늘려가며 테스트
for n in [1, 2, 4, 8]:
    concurrent_pixel_test(n)
    time.sleep(1)


  1개 동시 요청 (실제 추론)
2026-10-29 14:44:24 INFO     [ml_api] POST /predict/pixels → 200 (0.057s)
INFO:     127.0.0.1:14300 - "POST /predict/pixels HTTP/1.1" 200 OK
  요청 #1: 2.12초 (HTTP 200)
  전체: 2.12초

  2개 동시 요청 (실제 추론)
2026-10-29 14:44:27 INFO     [ml_api] POST /predict/pixels → 200 (0.002s)
INFO:     127.0.0.1:14303 - "POST /predict/pixels HTTP/1.1" 200 OK
2026-10-29 14:44:27 INFO     [ml_api] POST /predict/pixels → 200 (0.005s)
INFO:     127.0.0.1:14304 - "POST /predict/pixels HTTP/1.1" 200 OK
  요청 #1: 2.03초 (HTTP 200)
  요청 #2: 2.03초 (HTTP 200)
  전체: 2.03초

  4개 동시 요청 (실제 추론)
2026-10-29 14:44:30 INFO     [ml_api] POST /predict/pixels → 200 (0.007s)
INFO:     127.0.0.1:14310 - "POST /predict/pixels HTTP/1.1" 200 OK
2026-10-29 14:44:30 INFO     [ml_api] POST /predict/pixels → 200 (0.008s)
INFO:     127.0.0.1:14309 - "POST /predict/pixels HTTP/1.1" 200 OK
2026-10-29 14:44:30 INFO     [ml_api] POST /predict/pixels → 200 (0.01s)
INFO:     127.0.0.1:14311 - "POST /predict/pixels HTTP/1.1" 

---

### 6.3 에러 핸들링 동작 확인

In [4]:
print("=" * 50)
print("  에러 핸들링 테스트")
print("=" * 50)

# 정상 요청
image, label = test_dataset[0]
pixels = (np.array(image) / 255.0).tolist()
resp = requests.post("http://localhost:8000/predict/pixels", json={"pixels": pixels})
print(f"\n[정상 요청] 상태: {resp.status_code}, 예측: {resp.json()['predicted_class']}")

# 잘못된 픽셀 크기
resp = requests.post(
    "http://localhost:8000/predict/pixels",
    json={"pixels": [[0.0] * 14 for _ in range(14)]}
)
print(f"[잘못된 크기] 상태: {resp.status_code}")

# 잘못된 Base64
resp = requests.post(
    "http://localhost:8000/predict/image",
    json={"image_base64": "not_valid!!!"}
)
print(f"[잘못된 Base64] 상태: {resp.status_code}, 에러: {resp.json().get('detail', 'N/A')}")

# 헬스체크
resp = requests.get("http://localhost:8000/health")
print(f"[헬스체크] 상태: {resp.status_code}, 응답: {resp.json()}")

  에러 핸들링 테스트
2026-10-29 14:44:56 INFO     [ml_api] POST /predict/pixels → 200 (0.002s)
INFO:     127.0.0.1:6457 - "POST /predict/pixels HTTP/1.1" 200 OK

[정상 요청] 상태: 200, 예측: 7
2026-10-29 14:44:58 WARNING  [ml_api] POST /predict/pixels → 422 (0.008s)
INFO:     127.0.0.1:6459 - "POST /predict/pixels HTTP/1.1" 422 Unprocessable Entity
[잘못된 크기] 상태: 422
2026-10-29 14:45:00 WARNING  [ml_api] POST /predict/image → 400 (0.127s)
INFO:     127.0.0.1:6461 - "POST /predict/image HTTP/1.1" 400 Bad Request
[잘못된 Base64] 상태: 400, 에러: 이미지 처리 실패: cannot identify image file <_io.BytesIO object at 0x000002ABBFD31E90>
2026-10-29 14:45:02 INFO     [ml_api] GET /health → 200 (0.001s)
INFO:     127.0.0.1:5755 - "GET /health HTTP/1.1" 200 OK
[헬스체크] 상태: 200, 응답: {'status': 'healthy', 'model_loaded': True}


> 모든 비정상 요청이 **서버를 죽이지 않고**, 적절한 상태 코드와 메시지를 반환합니다.

---

### 6.4 프로젝트 구조 확인

```
model-serving-course/
├── app/
│   ├── error_handlers.py          ← 5장: 글로벌 에러 핸들러
│   ├── logger_config.py           ← 5장: 로깅 설정
│   ├── main_final.py              ← 6장: 최종 서버
│   ├── middleware.py              ← 5장: 요청/응답 로깅 미들웨어
│   ├── model_utils.py             ← REST API 노트: 모델 유틸리티
│   └── schemas.py                 ← FastAPI 기초 노트: Pydantic 스키마
├── models/
│   └── mnist_state_dict.pth
├── .gitignore
└── requirements.txt
```

## 7. LLM 서빙으로 넓히기 : 생성 직렬화, 스트리밍, 취소

> **학습 목표**
> - GPU 한 장을 지키도록 LLM 생성을 한 번에 하나씩 들여보낼 수 있습니다.  
> - SSE 스트리밍으로 첫 청크까지의 시간을 줄이고 잴 수 있습니다.  
> - 클라이언트가 떠나거나 예산을 넘으면 생성을 멈추게 할 수 있습니다.  
> - 모델을 올리는 동안 헬스체크가 `loading`이라고 답하게 할 수 있습니다.

6.2절에서 MNIST 추론 요청 하나는 0.1초도 걸리지 않았습니다. LLM은 답 하나를 토큰 수십\~수백 개로 몇 초에 걸쳐 만들고 그동안 GPU 메모리를 붙잡습니다. 그래서 LLM API 서버에는 6장에 없던 장치가 더 필요합니다.

재려는 것은 API 서버의 구조이므로 모델은 토큰 하나를 만드는 시간만 흉내 내는 가짜를 씁니다. 서버는 1\~6장의 파일을 덮어쓰지 않도록 새 파일 `app/stream_server.py`에 두고 포트도 따로 씁니다.

### 7.1 빈 포트 찾기와 버전 기록

6장 서버가 8000번을 쓰고 있으므로 이 장의 서버는 빈 포트를 찾아 씁니다. `pick_port`는 127.0.0.1에서 정해 둔 범위를 차례로 확인해 아무도 받지 않는 첫 포트를 돌려줍니다. 3.1절 서버 셀의 주석에 나온 "Address already in use"를 커널 재시작 없이 피하는 방법입니다.

In [5]:
import os
import socket
from importlib.metadata import version


def pick_port(start, end):
    """127.0.0.1에서 start부터 end까지 차례로 확인해 비어 있는 첫 포트를 돌려줍니다. 없으면 None입니다"""
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(("127.0.0.1", port)) != 0:    # 아무도 받지 않는 포트
                return port
    return None


# 포트를 코드에 직접 적지 않습니다. 6장 서버(8000)와 겹치지 않게 8048부터 찾습니다
START = int(os.environ.get("D48_PORT", "8048"))
PORT = pick_port(START, START + 9)
BASE = f"http://127.0.0.1:{PORT}"
print("이 장의 API 서버 주소 :", BASE if PORT else "빈 포트가 없습니다. D48_PORT를 바꾸십시오")

# 설치된 버전을 찍어 둡니다. 결과가 노트와 다르게 나오면 가장 먼저 볼 기록입니다
for name in ["fastapi", "starlette", "uvicorn", "anyio", "pydantic"]:
    print(f"{name:10} {version(name)}")

이 장의 API 서버 주소 : http://127.0.0.1:8048
fastapi    0.141.1
starlette  1.10.0
uvicorn    0.53.0
anyio      4.15.1
pydantic   2.14.1


### 7.2 가짜 LLM을 올린 API 서버와 준비 상태

`app/stream_server.py`는 5장의 `setup_logger`와 `register_error_handlers`를 그대로 씁니다. 5.4절의 요청 로깅 미들웨어는 달지 않았습니다. 이유는 7.5절의 더 알아보기에 있습니다.

In [6]:
%%writefile app/stream_server.py
"""LLM API 서버 연습 : 생성 직렬화, SSE 스트리밍, 취소, 예산, 준비 상태

모델은 가짜입니다. 토큰 하나를 만드는 데 걸리는 시간만 흉내 냅니다.
진짜 LLM으로 바꾸는 자리는 generate_tokens() 하나입니다.
"""
import asyncio
import json
import os
import time
from contextlib import aclosing, asynccontextmanager

import anyio
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field

from app.error_handlers import register_error_handlers
from app.logger_config import setup_logger

logger = setup_logger("ml_api")

# 상수를 코드에 직접 적지 않고 환경 변수에서 읽습니다
TOKENS_PER_SECOND = float(os.environ.get("D48_TPS", "20"))      # 생성 속도
ANSWER_TOKENS = int(os.environ.get("D48_OUT_TOKENS", "60"))   # 답 하나의 토큰 수 (약 3초)
LOAD_SECONDS = float(os.environ.get("D48_LOAD_SECONDS", "3"))    # 모델을 올리는 시간
GPU_SLOTS = int(os.environ.get("D48_GPU_SLOTS", "1"))            # 동시에 생성하는 요청 수
MAX_TOKENS = int(os.environ.get("D48_MAX_TOKENS", "500"))        # 토큰 예산 기본값
WALL_CLOCK = float(os.environ.get("D48_WALL_CLOCK", "30"))       # 시간 예산 기본값 (초)

STATE = {"준비됨": False}
STATS = {"만든 토큰": 0}
gpu_limiter = anyio.CapacityLimiter(GPU_SLOTS)


def generate_tokens(question):
    """가짜 LLM. 토큰을 하나씩 내놓는 동기 제너레이터입니다"""
    for i in range(ANSWER_TOKENS):
        time.sleep(1 / TOKENS_PER_SECOND)        # 토큰 하나를 계산하는 시간
        STATS["만든 토큰"] += 1
        yield f"토큰{i} "


async def load_model():
    await asyncio.sleep(LOAD_SECONDS)             # 진짜 서버라면 여기서 모델을 GPU에 올립니다
    STATE["준비됨"] = True
    logger.info("모델 준비 완료")


@asynccontextmanager
async def lifespan(app):
    task = asyncio.create_task(load_model())      # 뜰 때 : 모델 올리기를 뒤에서 시작합니다
    yield                                          # 이 사이에 요청을 받습니다
    task.cancel()                                  # 내릴 때 : 정리합니다
    logger.info("서버를 내립니다")


app = FastAPI(title="LLM Streaming API", lifespan=lifespan)
register_error_handlers(app)


class AskRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=500)
    max_tokens: int | None = Field(default=None, ge=1, le=2000)          # 이 요청의 토큰 예산
    deadline_seconds: float | None = Field(default=None, gt=0, le=300)   # 이 요청의 시간 예산


def new_budget(req):
    """요청이 예산을 주지 않으면 서버 기본값을 씁니다"""
    return {"max_tokens": req.max_tokens or MAX_TOKENS,
            "deadline": req.deadline_seconds or WALL_CLOCK}


async def guarded_tokens(question, request, budget, report):
    """한 번에 GPU_SLOTS 건만 통과시키고, 예산과 클라이언트 이탈을 보며 토큰을 내놓습니다"""
    it = generate_tokens(question)
    done = object()
    count = 0
    queued_at = started = time.time()
    report["정지 사유"] = "밖에서 끊김"
    try:
        async with gpu_limiter:                    # 슬롯이 빌 때까지 여기서 기다립니다
            started = time.time()
            report["대기 초"] = round(started - queued_at, 2)
            while True:
                # next() 한 번(토큰 하나)만 스레드에서 돌립니다
                token = await anyio.to_thread.run_sync(next, it, done)
                if token is done:
                    report["정지 사유"] = "완료"
                    break
                count += 1
                yield token
                if count >= budget["max_tokens"]:
                    report["정지 사유"] = "토큰 예산 소진"
                    break
                if time.time() - started >= budget["deadline"]:
                    report["정지 사유"] = "시간 예산 소진"
                    break
                if await request.is_disconnected():
                    report["정지 사유"] = "클라이언트 이탈"
                    break
    finally:
        it.close()                                 # 생성을 실제로 멈춥니다
        report["토큰"] = count
        report["생성 초"] = round(time.time() - started, 2)
        logger.info(f"생성 종료 : {report}")


def sse(event, data):
    """SSE 한 청크. 빈 줄 하나로 끝납니다"""
    return f"event: {event}\ndata: {json.dumps(data, ensure_ascii=False)}\n\n"


def not_ready():
    return JSONResponse(status_code=503, content={"error": "모델을 올리는 중입니다"})


@app.get("/health")
async def health():
    return {"status": "healthy" if STATE["준비됨"] else "loading"}


@app.get("/stats")
async def stats():
    return dict(STATS)


@app.post("/ask")
async def ask(req: AskRequest, request: Request):
    """답을 다 만든 뒤 한 번에 돌려줍니다"""
    if not STATE["준비됨"]:
        return not_ready()
    report, parts = {}, []
    async with aclosing(guarded_tokens(req.question, request, new_budget(req), report)) as stream:
        async for token in stream:
            parts.append(token)
    return {"answer": "".join(parts), **report}


@app.post("/ask/sse")
async def ask_sse(req: AskRequest, request: Request):
    """만들어지는 대로 SSE로 스트리밍합니다"""
    if not STATE["준비됨"]:
        return not_ready()
    report = {}

    async def events():
        yield sse("start", {"question": req.question})
        try:
            async with aclosing(guarded_tokens(req.question, request, new_budget(req), report)) as stream:
                async for token in stream:
                    yield sse("token", {"text": token})
        except Exception as exc:
            # 200이 이미 나갔으므로 오류도 본문 안의 청크로 알립니다
            logger.error(f"생성 중 오류 : {exc}")
            yield sse("error", {"error": "생성 중 오류가 발생했습니다"})
            return
        yield sse("done", report)

    return StreamingResponse(events(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no"})   # 앞단 프록시가 모아 두지 않게

Writing app/stream_server.py


**눈여겨볼 곳**  
- `generate_tokens` : 진짜 LLM으로 바꾸는 자리입니다. 토큰 하나에 0.05초씩 60개를 내놓으므로 답 하나에 약 3초가 걸립니다  
- `lifespan` : 6.1절의 `@app.on_event("startup")` 자리를 대신합니다. `yield` 앞은 뜰 때, 뒤는 내릴 때 실행됩니다  
- `load_model` : 모델 올리기를 뒤에서 돌리므로 서버는 곧바로 요청을 받고 그동안 `/health`가 `loading`이라고 답합니다

6.1절 최종 서버의 `/health`에도 `loading` 분기가 있습니다. 그런데 uvicorn은 `startup`이 끝난 뒤에야 요청을 받기 시작하므로 모델을 올리는 동안 온 헬스체크는 `loading` 대신 연결 거부를 받습니다. 로드밸런서는 헬스체크에 답이 없는 서버를 다운으로 판단합니다(3.4절).

In [7]:
import threading
import time

import requests
import uvicorn

config = uvicorn.Config("app.stream_server:app", host="127.0.0.1", port=PORT, log_level="warning")
stream_server = uvicorn.Server(config)
stream_thread = threading.Thread(target=stream_server.run, daemon=True)
stream_thread.start()


def wait_ready(timeout=60):
    """/health 가 healthy 가 될 때까지 0.5초마다 묻고, 상태가 바뀔 때만 찍습니다"""
    t0, last = time.time(), None
    while time.time() - t0 < timeout:
        try:
            status = requests.get(BASE + "/health", timeout=1).json()["status"]
        except requests.ConnectionError:
            status = "응답 없음"                  # 아직 요청을 받지 않습니다
        if status != last:
            note = ""
            if status == "loading":               # 준비 중에 질문을 보내 봅니다
                code = requests.post(BASE + "/ask", json={"question": "연차는 며칠입니까"}).status_code
                note = f"   (이때 /ask 응답 코드 {code})"
            print(f"{time.time() - t0:4.1f}초  /health : {status}{note}")
            last = status
        if status == "healthy":
            return True
        time.sleep(0.5)
    return False


if not wait_ready():
    print("60초 안에 준비되지 않았습니다. 셀 출력에 찍힌 오류를 보십시오")

 0.5초  /health : loading   (이때 /ask 응답 코드 503)
2026-10-29 15:00:00 INFO     [ml_api] 모델 준비 완료
 3.0초  /health : healthy


서버는 1초 안에 요청을 받기 시작했고 약 3초 동안 `loading`이었습니다. 그동안 `/ask`는 503을 돌려줬습니다. 503은 서버가 살아 있지만 아직 요청을 처리할 수 없다는 상태 코드입니다.

### 7.3 생성 직렬화 : GPU 한 장에는 한 번에 하나씩

4.5절은 GPU 추론의 스레드풀을 1\~2로 두라고 했습니다. LLM은 답 하나를 만드는 몇 초 내내 GPU 메모리(KV 캐시)를 잡고 있으므로 스레드 수보다 **생성 한 건 전체**를 단위로 막아야 합니다. `guarded_tokens`의 `async with gpu_limiter:`가 그 문지기입니다.

- `anyio.CapacityLimiter(N)` : 동시에 N건만 `async with` 안으로 들여보내고 나머지는 슬롯이 빌 때까지 기다리게 합니다  
- `anyio.to_thread.run_sync` : 4장의 `run_in_executor`처럼 동기 함수를 스레드에서 돌립니다. FastAPI가 일반 `def` 엔드포인트를 돌릴 때 쓰는 함수이고 3.5절의 "보통 40개"가 이 함수의 기본 한도입니다  
- 생성 직렬화 : 생성을 한 번에 하나씩만 들여보내는 방식입니다(`GPU_SLOTS=1`). REST API 노트의 모델 직렬화(파일로 저장)와는 다른 뜻입니다

In [8]:
from concurrent.futures import ThreadPoolExecutor


def ask_many(n):
    """같은 질문 n개를 동시에 /ask 로 보내고, 요청마다 걸린 초를 돌려줍니다"""
    def one(i):
        t0 = time.time()
        requests.post(BASE + "/ask", json={"question": "연차는 며칠입니까"}, timeout=120)
        return round(time.time() - t0, 1)

    with ThreadPoolExecutor(max_workers=n) as ex:
        return sorted(ex.map(one, range(n)))


print("동시 3개, GPU_SLOTS=1 :", ask_many(3))

2026-10-29 15:35:43 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '대기 초': 0.0, '토큰': 60, '생성 초': 3.06}
2026-10-29 15:35:47 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '대기 초': 3.05, '토큰': 60, '생성 초': 3.05}
2026-10-29 15:35:50 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '대기 초': 6.11, '토큰': 60, '생성 초': 3.05}
동시 3개, GPU_SLOTS=1 : [3.1, 6.1, 9.2]


세 요청이 3.1초, 6.1초, 9.2초에 끝났습니다. 4.2절의 `v3-executor`가 동시 3개를 모두 3초에 끝낸 것과 달리 일부러 줄을 세운 결과입니다.

진짜 LLM이라면 세 요청을 함께 돌릴 때 KV 캐시도 세 벌이 필요하고 메모리가 모자라면 세 요청이 모두 실패할 수 있습니다. 생성 직렬화는 처리량을 내주고 예측할 수 있는 메모리를 얻습니다. `D48_GPU_SLOTS`를 올리면 줄이 풀립니다.

줄이 얼마나 길어지면 새 요청을 거절해야 하는지, 기다리는 사람에게 몇 번째라고 알려줄지는 나중에 배웁니다. 오늘은 줄을 세우는 데까지입니다.

### 7.4 SSE 스트리밍과 첫 청크까지의 시간

`/ask`는 답을 다 만든 뒤에 한 번에 보내므로 클라이언트는 그동안 빈 화면을 봅니다. `/ask/sse`는 토큰을 만들어지는 대로 SSE(Server-Sent Events)로 보냅니다. `timing`은 두 엔드포인트의 첫 청크까지의 시간과 전체 시간을 잽니다.

In [9]:
def timing(path):
    """첫 청크(첫 토큰)까지의 시간과 전체 시간을 잽니다"""
    t0, first, lines = time.time(), None, []
    with requests.post(BASE + path, json={"question": "연차는 며칠입니까"},
                       stream=True, timeout=60) as r:
        for line in r.iter_lines(chunk_size=None, decode_unicode=True):
            if first is None and "토큰0" in line:
                first = time.time() - t0
            lines.append(line)
    return first, time.time() - t0, lines


for path in ["/ask", "/ask/sse"]:
    first, total, lines = timing(path)
    print(f"{path:9} 첫 청크까지 {first:.2f}초   전체 {total:.2f}초")

print()
print("\n".join(lines[:5] + ["..."] + lines[-3:]))     # /ask/sse 로 받은 줄의 앞과 끝

2026-10-29 15:36:20 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '대기 초': 0.0, '토큰': 60, '생성 초': 3.05}
/ask      첫 청크까지 3.06초   전체 3.06초
2026-10-29 15:36:23 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '대기 초': 0.0, '토큰': 60, '생성 초': 3.06}
/ask/sse  첫 청크까지 0.05초   전체 3.06초

event: start
data: {"question": "연차는 며칠입니까"}

event: token
data: {"text": "토큰0 "}
...
event: done
data: {"정지 사유": "완료", "대기 초": 0.0, "토큰": 60, "생성 초": 3.06}



전체 시간은 두 엔드포인트 모두 약 3초인데 첫 청크까지의 시간은 약 3초와 0.05초로 갈렸습니다. 생성 속도는 그대로이고 사용자가 빈 화면을 보는 시간만 줄었습니다.

SSE는 `event:` 줄에 청크의 종류를, `data:` 줄에 내용을 적고 빈 줄 하나로 청크 하나를 끝냅니다. 그래서 답의 토큰(`token`)과 본문이 아닌 알림(`start`, `done`)을 섞지 않고 한 연결로 보낼 수 있습니다.

**첫 청크가 나가는 순간 상태 코드 200이 굳습니다.** 그 뒤에 생성이 실패하면 500으로 바꿀 수 없으므로 `ask_sse`는 오류를 `event: error` 청크로 본문에 실어 보냅니다. 클라이언트는 `done` 대신 `error`가 오는 것으로 실패를 압니다.

### 7.5 취소 : 클라이언트가 떠나면 생성 멈추기

클라이언트가 답을 기다리다 창을 닫아도 서버가 모르면 아무도 읽지 않을 답을 끝까지 만들고 그동안 7.3절의 슬롯을 차지합니다. `cut`은 0.8초 만에 연결을 끊고 그 요청 때문에 서버가 만든 토큰 수를 셉니다.

In [10]:
def made_tokens():
    return requests.get(BASE + "/stats").json()["만든 토큰"]


def cut(path, after=0.8):
    """after초 뒤에 연결을 끊고, 그 요청 때문에 서버가 만든 토큰 수를 돌려줍니다"""
    before, t0 = made_tokens(), time.time()
    try:
        with requests.post(BASE + path, json={"question": "연차는 며칠입니까"},
                           stream=True, timeout=after) as r:
            for line in r.iter_lines(chunk_size=None):
                if time.time() - t0 >= after:
                    break                        # with 블록을 나가면 연결이 닫힙니다
    except requests.exceptions.ReadTimeout:
        pass                                     # /ask 는 after초 안에 아무것도 오지 않아 여기서 끊깁니다
    time.sleep(3.5)                              # 서버가 끝까지 만든다면 이 사이에 다 만듭니다
    return made_tokens() - before


for path in ["/ask/sse", "/ask"]:
    print(f"{path:9} 0.8초에 끊었을 때 서버가 만든 토큰 : {cut(path)}개 (끝까지 가면 60개)")

2026-10-29 15:36:39 INFO     [ml_api] 생성 종료 : {'정지 사유': '클라이언트 이탈', '대기 초': 0.0, '토큰': 17, '생성 초': 0.87}
/ask/sse  0.8초에 끊었을 때 서버가 만든 토큰 : 17개 (끝까지 가면 60개)
2026-10-29 15:36:43 INFO     [ml_api] 생성 종료 : {'정지 사유': '클라이언트 이탈', '대기 초': 0.0, '토큰': 16, '생성 초': 0.81}
/ask      0.8초에 끊었을 때 서버가 만든 토큰 : 16개 (끝까지 가면 60개)


두 엔드포인트 모두 끝까지 가면 60개인 토큰을 16개 안팎에서 멈췄습니다. 서버 로그의 `생성 종료` 줄에서 멈춘 길을 보십시오.

- `/ask/sse` : `정지 사유`는 `밖에서 끊김`이거나 `클라이언트 이탈`입니다. 스트리밍 응답(`StreamingResponse`)이 연결을 따로 지켜보다가 끊기면 제너레이터를 멈추는데 `guarded_tokens`의 확인이 먼저 잡을 때도 있습니다  
- `/ask` : `정지 사유`가 `클라이언트 이탈`입니다. 단일 응답에는 그런 감시가 없어서 서버가 직접 묻지 않으면 클라이언트가 떠난 것을 알 기회가 없습니다. 그래서 `guarded_tokens`가 토큰마다 `await request.is_disconnected()`로 묻습니다. 이 확인을 빼면 60개를 끝까지 만듭니다

**감싸면 안쪽은 자동으로 닫히지 않습니다.** `ask`와 `ask_sse`는 `guarded_tokens`를 한 겹 감싸서 씁니다. `contextlib.aclosing`으로 묶어 두면 바깥 반복이 어떻게 끝나든 곧바로 안쪽의 `finally`(`it.close()`와 슬롯 반환)가 실행됩니다.

🔎 **더 알아보기**
`aclosing`을 빼고 `ask`의 반복이 다섯 토큰 뒤에 `break`하도록 고치면 안쪽 `guarded_tokens`는 곧바로 닫히지 않습니다. 나중에 파이썬이 다른 작업(task)에서 닫으면서 `CapacityLimiter`의 슬롯을 돌려주지 못해 `RuntimeError`가 납니다(anyio 4.13.0에서 확인). 슬롯이 돌아오지 않으므로 그 뒤의 `/ask`는 기다리기만 하고 답을 받지 못합니다.

🔎 **더 알아보기**
5.4절의 `RequestLoggingMiddleware`는 `BaseHTTPMiddleware`로 만들었습니다. 이 미들웨어를 `app/stream_server.py`에 달면 SSE 연결이 끊겨도 `guarded_tokens`의 `finally`가 제때 실행되지 않습니다. starlette 1.0.0과 1.7.0에서 연결을 끊고 4초를 기다려도 실행되지 않았고 미들웨어가 없으면 끊자마자 실행됐습니다.

스트리밍 서버의 요청 로깅은 `BaseHTTPMiddleware` 대신 순수 ASGI 미들웨어로 씁니다. 서빙 프로젝트에서 그 코드를 씁니다.

### 7.6 토큰 예산과 시간 예산

생성에는 상한을 겁니다. `max_tokens`는 요청 하나가 만들 수 있는 토큰 수이고 `deadline_seconds`는 생성에 쓸 수 있는 초입니다. 요청마다 값을 줄 수 있고 주지 않으면 서버 기본값(`D48_MAX_TOKENS`, `D48_WALL_CLOCK`)을 씁니다.

In [11]:
for budget in [{}, {"max_tokens": 10}, {"deadline_seconds": 1.0}]:
    r = requests.post(BASE + "/ask", json={"question": "연차는 며칠입니까", **budget}, timeout=60)
    body = r.json()
    print(f"{str(budget):26} 상태 {r.status_code}  정지 사유 '{body['정지 사유']}'  토큰 {body['토큰']}개")

2026-10-29 15:37:01 INFO     [ml_api] 생성 종료 : {'정지 사유': '완료', '대기 초': 0.0, '토큰': 60, '생성 초': 3.05}
{}                         상태 200  정지 사유 '완료'  토큰 60개
2026-10-29 15:37:01 INFO     [ml_api] 생성 종료 : {'정지 사유': '토큰 예산 소진', '대기 초': 0.0, '토큰': 10, '생성 초': 0.51}
{'max_tokens': 10}         상태 200  정지 사유 '토큰 예산 소진'  토큰 10개
2026-10-29 15:37:02 INFO     [ml_api] 생성 종료 : {'정지 사유': '시간 예산 소진', '대기 초': 0.0, '토큰': 20, '생성 초': 1.02}
{'deadline_seconds': 1.0}  상태 200  정지 사유 '시간 예산 소진'  토큰 20개


예산에 걸린 응답도 상태 코드는 200입니다. 예산 소진은 정해 둔 정상 경로라서 오류로 돌려주지 않고 `정지 사유`를 함께 보냅니다. 클라이언트는 `정지 사유`로 답이 짧은 이유를 압니다.

### 7.7 스트리밍 서버의 약속과 정리

게이트웨이 노트와 서빙 프로젝트의 API 서버도 `app/stream_server.py`와 같은 약속을 지킵니다.

<div style="font-size: 18px; line-height: 1.6;">

| 엔드포인트 | 하는 일 | 상태 코드 |
|---|---|---|
| `GET /health` | 준비 상태를 답합니다 | 200. `status`가 `loading` 또는 `healthy` |
| `POST /ask` | 답을 한 번에 돌려줍니다 | 200, 준비 중 503, 입력 오류 422 |
| `POST /ask/sse` | `start`, `token`, `done` 청크로 스트리밍합니다. 실패하면 `error` 청크 | 200, 준비 중 503, 입력 오류 422 |

</div>

In [12]:
# 이 장의 서버만 내립니다. lifespan 의 yield 아래(내릴 때)가 실행됩니다
stream_server.should_exit = True
stream_thread.join(timeout=10)
print("포트", PORT, "정리 완료" if pick_port(PORT, PORT) == PORT else "아직 사용 중")

2026-10-29 15:37:07 INFO     [ml_api] 서버를 내립니다
포트 8048 정리 완료


### 체크포인트

1. 6.1절 최종 서버의 `/health`에는 `loading` 분기가 있는데 모델을 올리는 동안 `loading`을 볼 수 없는 이유는 무엇입니까?  
2. GPU 한 장짜리 LLM API 서버에서 `GPU_SLOTS=1`로 생성을 직렬화하면 무엇을 잃고 무엇을 얻습니까?  
3. 전체 시간이 같은데도 SSE 스트리밍이 낫다고 말하는 근거는 무엇입니까?  
4. 단일 응답(`/ask`)에서 클라이언트가 떠난 것을 알려면 서버가 무엇을 해야 합니까?  
5. 예산에 걸려 답이 짧아진 응답에 500을 주면 안 되는 이유는 무엇입니까?

## 최종 체크포인트

```
Q1. 동기 서버에서 3초 걸리는 추론을 3명이 동시에 요청하면 총 몇 초 걸립니까?
Q2. time.sleep(3)과 await asyncio.sleep(3)의 핵심 차이는?
Q3. async def 안에서 동기 블로킹 코드를 실행하면 왜 헬스체크까지 영향받습니까?
Q4. run_in_executor가 이벤트 루프 블로킹을 방지하는 원리는?
Q5. 글로벌 Exception Handler를 사용하는 이유는?
Q6. 클라이언트에게 스택 트레이스를 노출하면 안 되는 이유는?
```

---

### 요약

```
오늘 한 일:
  - 동기/비동기의 차이와 모델 추론에서의 의미를 이해했습니다.
  - 동기 추론이 서버를 멈추는 문제를 재현하고 헬스체크 차단까지 관찰했습니다.
  - run_in_executor 패턴으로 문제를 해결했습니다.
  - 글로벌 에러 핸들러와 로깅으로 서버 안정성을 높였습니다.
  - 동시 요청 테스트로 동작을 확인했습니다.

내일 (인증과 화면 노트):
  - Streamlit으로 웹 UI를 만들고 FastAPI와 연결합니다.
  - API Key 인증과 파일 업로드를 더합니다.
```

**7장 요약**  
- LLM API 서버는 생성을 한 번에 하나씩 들여보내 GPU 메모리를 지킵니다(`CapacityLimiter`)  
- SSE로 보내면 전체 시간은 같아도 첫 청크까지의 시간이 약 3초에서 0.05초로 줄었습니다  
- 클라이언트가 떠나면 생성을 멈추고 예산을 넘으면 `정지 사유`와 함께 200으로 답합니다  
- 모델을 올리는 동안 헬스체크는 `loading`이고 `/ask`와 `/ask/sse`는 503을 돌려줍니다

### 제출

다음 내역을 MD 파일로 기록, 깃헙에 업로드하여 링크로 제출하시기 바랍니다

1. 섹션 1.5 수행내역 캡쳐  
2. 섹션 2, 3 셀 출력  
3. 섹션 5 수행내역 캡쳐  
4. 각 섹션 체크포인트의 답변

7장을 했다면 7.4절(첫 청크까지의 시간)과 7.5절(취소) 셀 출력도 함께 올리십시오.

수고하셨습니다!

## Reference

**lifespan과 `on_event`**  
출처 : FastAPI, [Lifespan Events](https://fastapi.tiangolo.com/advanced/events/), 2026.10.07 확인 ("Alternative Events (deprecated)")

**스레드와 동시 통과 제한**  
출처 : AnyIO, [Working with threads](https://anyio.readthedocs.io/en/stable/threads.html), 2026.10.07 확인 (`to_thread.run_sync`의 기본 한도 40)

**스트리밍, 취소, 미들웨어**  
출처 : WHATWG, [HTML Living Standard : Server-sent events](https://html.spec.whatwg.org/multipage/server-sent-events.html), 2026.10.07 확인 (SSE 형식의 정본)  
출처 : Python, [contextlib.aclosing](https://docs.python.org/3/library/contextlib.html#contextlib.aclosing), 2026.10.07 확인  
출처 : Starlette, [Middleware](https://www.starlette.io/middleware/), 2026.10.07 확인 (순수 ASGI 미들웨어)  
출처 : Starlette 1.0.0 소스, `starlette/responses.py`의 `StreamingResponse.__call__` (연결 끊김 감시)